In [0]:
import os
import requests
import pandas as pd
import time
from datetime import date, timedelta

BASE_URL = "https://www.consumerfinance.gov/data-research/consumer-complaints/search/api/v1/"
HEADERS = {"Accept": "application/json"}


def find_repo_root():
    """Walk up from the current folder until a .git folder is found (the repo root)."""
    path = os.getcwd()
    while path != os.path.dirname(path):
        if os.path.isdir(os.path.join(path, ".git")):
            return path
        path = os.path.dirname(path)
    return os.getcwd()  # fallback if no .git is found


# Output goes to <repo root>/data/samples, NOT the Notebooks folder.
# Change "samples" to "sample" here if that is your folder name.
OUTPUT_DIR = os.path.join(find_repo_root(), "data", "samples")
os.makedirs(OUTPUT_DIR, exist_ok=True)


def fetch_page(date_min, date_max, frm, page_size=100, retries=3):
    params = {
        "size": page_size,
        "frm": frm,
        "date_received_min": date_min,
        "date_received_max": date_max,
        "sort": "created_date_desc",
    }
    for attempt in range(retries):
        try:
            response = requests.get(BASE_URL, params=params, headers=HEADERS, timeout=30)
            response.raise_for_status()
            return response.json().get("hits", {}).get("hits", [])
        except requests.exceptions.RequestException as e:
            print(f"    retry {attempt + 1}/{retries} after error: {e}")
            time.sleep(2)
    return []


def fetch_by_day(start_date, end_date, target_rows, page_size=100):
    all_rows = []
    current = start_date

    while current <= end_date and len(all_rows) < target_rows:
        day_str = current.isoformat()
        next_day_str = (current + timedelta(days=1)).isoformat()

        frm = 0
        while True:
            hits = fetch_page(day_str, next_day_str, frm, page_size)
            if not hits:
                break
            all_rows.extend([h["_source"] for h in hits])
            frm += page_size
            if frm >= 500:  # cap pagination depth per day, move to next day instead
                break
            time.sleep(0.2)

        print(f"  {day_str}: total so far {len(all_rows)} / {target_rows}")
        current += timedelta(days=1)

    return pd.DataFrame(all_rows[:target_rows])


# --- Raw data sample (disabled) ---
# print("Pulling raw data sample...")
# raw_sample = fetch_by_day(date(2026, 6, 1), date(2026, 6, 1), target_rows=20)
# raw_sample.to_csv(os.path.join(OUTPUT_DIR, "raw_data_sample.csv"), index=False)
# print(f"Raw data sample saved: {len(raw_sample)} rows\n")

# --- Full load (disabled) ---
# print("Pulling full load...")
# full_load = fetch_by_day(date(2026, 6, 1), date(2026, 8, 31), target_rows=30000)
# full_load.to_csv(os.path.join(OUTPUT_DIR, "full_load_sample.csv"), index=False)
# print(f"Full load saved: {len(full_load)} rows\n")

# --- Incremental load (active) ---
print("Pulling incremental load...")
incremental_load = fetch_by_day(date(2026, 9, 1), date(2026, 9, 7), target_rows=6000)
out_path = os.path.join(OUTPUT_DIR, "incremental_load_sample.csv")
incremental_load.to_csv(out_path, index=False)
print(f"Incremental load saved: {len(incremental_load)} rows -> {out_path}")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType

spark = SparkSession.builder \
    .appName("ConsumerComplaintAnalytics") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")

DATA_BASE = "/Workspace/Users/ssalmanaali80@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples"

complaint_schema = StructType([
    StructField("complaint_id", StringType(), True),
    StructField("date_received", DateType(), True),
    StructField("date_sent_to_company", DateType(), True),
    StructField("product", StringType(), True),
    StructField("sub_product", StringType(), True),
    StructField("issue", StringType(), True),
    StructField("sub_issue", StringType(), True),
    StructField("company", StringType(), True),
    StructField("state", StringType(), True),
    StructField("zip_code", StringType(), True),
    StructField("submitted_via", StringType(), True),
    StructField("company_response", StringType(), True),
    StructField("timely", StringType(), True),
    StructField("tags", StringType(), True),
])

full_load_df = spark.read.option("header", "true").schema(complaint_schema).csv(f"{DATA_BASE}/full_load_sample.csv")
incremental_df = spark.read.option("header", "true").schema(complaint_schema).csv(f"{DATA_BASE}/incremental_load_sample.csv")
raw_df = spark.read.option("header", "true").csv(f"{DATA_BASE}/raw_data_sample.csv")

# spark.stop()

In [0]:
raw_df.show()

In [0]:
import pandas as pd
df = pd.read_csv("/Workspace/Users/ssalmanaali80@gmail.com/Consumer-Financial-Complaint-Analytics/data/samples/incremental_load_sample.csv", nrows=3)
df